In [1]:
%pip install openlineage-sql

Note: you may need to restart the kernel to use updated packages.


После установки библиотеки в Jupyter Notebook часто срабатывает кеширование путей. Если просто запустить import, ошибка может повториться.В верхнем меню Jupyter нажмите Kernel (Ядро).Выберите Restart Kernel (Перезапустить ядро).После этого ячейка с импортом отработает корректно:

In [10]:
import uuid
from datetime import datetime, timezone
from openlineage.client.run import RunEvent, RunState, Run, Job, InputDataset, OutputDataset
from openlineage.client.facet import SqlJobFacet
from openlineage_sql import parse

In [12]:

# 1. Определение нашего представления (View Definition)
# SQL содержит две исходные таблицы (users и orders) и одно результирующее представление
view_definition_sql = """
CREATE VIEW analytics.active_user_orders AS
SELECT 
    u.id AS user_id,
    u.name,
    count(o.id) as total_orders
FROM prod_tables.users u
JOIN prod_tables.orders o ON u.id = o.user_id
WHERE u.status = 'active'
GROUP BY u.id, u.name;
"""

# 2. Используем openlineage-sql для автоматического разбора запроса
# В dialect можно передать "postgres", "snowflake", "bigquery", "mysql" и т.д.
metadata = parse(sql=[view_definition_sql], dialect="postgres")

In [13]:
print(metadata)

{"in_tables": [DbTableMeta(DbTableMeta { database: None, schema: Some("prod_tables"), name: "orders", quote_style: Some(QuoteStyle { database: None, schema: None, name: None }), provided_namespace: false, provided_field_schema: false }), DbTableMeta(DbTableMeta { database: None, schema: Some("prod_tables"), name: "users", quote_style: Some(QuoteStyle { database: None, schema: None, name: None }), provided_namespace: false, provided_field_schema: false })], "out_tables": [DbTableMeta(DbTableMeta { database: None, schema: Some("analytics"), name: "active_user_orders", quote_style: Some(QuoteStyle { database: None, schema: None, name: None }), provided_namespace: false, provided_field_schema: false })] "column_lineage": [ColumnLineage { descendant: ColumnMeta(ColumnMeta { origin: None, name: "name" }), lineage: [ColumnMeta(ColumnMeta { origin: Some(DbTableMeta { database: None, schema: Some("prod_tables"), name: "users", quote_style: Some(QuoteStyle { database: None, schema: None, name: N

In [24]:
# 3. Формируем списки входных и выходных датасетов на основе распарсенного SQL
# По умолчанию парсер возвращает объекты DbTable (name и namespace)
MY_NAMESPACE = "postgres://prod-db-server:5432"

input_datasets = [
    InputDataset(namespace=MY_NAMESPACE, name=".".join([p for p in [db_table.database, db_table.schema, db_table.name] if p]))
    for db_table in metadata.in_tables
]



output_datasets = [
    OutputDataset(namespace=MY_NAMESPACE, name=".".join([p for p in [db_table.database, db_table.schema, db_table.name] if p]))
    for db_table in metadata.out_tables
]
print (input_datasets)
print (output_datasets)

[InputDataset(namespace='postgres://prod-db-server:5432', name='prod_tables.orders', facets={}, inputFacets={}), InputDataset(namespace='postgres://prod-db-server:5432', name='prod_tables.users', facets={}, inputFacets={})]
[OutputDataset(namespace='postgres://prod-db-server:5432', name='analytics.active_user_orders', facets={}, outputFacets={})]


In [25]:

# 4. Создаем метаданные самого процесса (Job) и запуска (Run)
job = Job(
    namespace="dwh_migrations",
    name="create_active_user_orders_view",
    facets={
        # Рекомендуется прикреплять исходный SQL-код к самому Job через специальный фасет
        "sql": SqlJobFacet(query=view_definition_sql)
    }
)


In [28]:
PRODUCER_URI = "https://github.com"
run = Run(
    runId=str(uuid.uuid4()),
    facets={}
)

# 5. Собираем финальный документ (RunEvent) со всеми связями
lineage_event = RunEvent(
    eventType=RunState.COMPLETE,  # Так как создание view обычно мгновенно, фиксируем COMPLETE
    eventTime=datetime.now(timezone.utc).isoformat(),
    run=run,
    job=job,
    inputs=input_datasets,
    outputs=output_datasets,
    producer=PRODUCER_URI
)



In [29]:
# 6. Печатаем результат в формате JSON (документ готов к отправке в Marquez / Airflow)

from openlineage.client.serde import Serde

# Сгенерировать JSON-строку (String)
json_string = Serde.to_json(lineage_event)
print(json_string)

# Или сгенерировать обычный Python-словарь (Dict)
# json_dict = Serde.to_dict(lineage_event)

{"eventTime": "2026-06-22T09:39:24.267794+00:00", "eventType": "COMPLETE", "inputs": [{"facets": {}, "inputFacets": {}, "name": "prod_tables.orders", "namespace": "postgres://prod-db-server:5432"}, {"facets": {}, "inputFacets": {}, "name": "prod_tables.users", "namespace": "postgres://prod-db-server:5432"}], "job": {"facets": {"sql": {"_producer": "https://github.com/OpenLineage/OpenLineage/tree/1.16.0/client/python", "_schemaURL": "https://raw.githubusercontent.com/OpenLineage/OpenLineage/main/spec/OpenLineage.json#/definitions/SqlJobFacet", "query": "\nCREATE VIEW analytics.active_user_orders AS\nSELECT \n    u.id AS user_id,\n    u.name,\n    count(o.id) as total_orders\nFROM prod_tables.users u\nJOIN prod_tables.orders o ON u.id = o.user_id\nWHERE u.status = 'active'\nGROUP BY u.id, u.name;\n"}}, "name": "create_active_user_orders_view", "namespace": "dwh_migrations"}, "outputs": [{"facets": {}, "name": "analytics.active_user_orders", "namespace": "postgres://prod-db-server:5432", 